In [1]:
import pandas as pd

train = pd.read_csv("../../Week1/Data/train.csv")

In [2]:
train.head()

,timestamp,value,label,KPI ID
0,1493568000,1.901639,0,02e99bd4f6cfb33f
1,1493568060,1.786885,0,02e99bd4f6cfb33f
2,1493568120,2.000000,0,02e99bd4f6cfb33f
3,1493568180,1.885246,0,02e99bd4f6cfb33f
4,1493568240,1.819672,0,02e99bd4f6cfb33f


In [3]:
df = train.copy()

In [4]:
df

,timestamp,value,label,KPI ID
0,1493568000,1.901639,0,02e99bd4f6cfb33f
1,1493568060,1.786885,0,02e99bd4f6cfb33f
2,1493568120,2.000000,0,02e99bd4f6cfb33f
3,1493568180,1.885246,0,02e99bd4f6cfb33f
4,1493568240,1.819672,0,02e99bd4f6cfb33f
...,...,...,...,...
2476310,1496895300,0.161922,1,88cf3a776ba00e7c
2476311,1496895360,0.162297,1,88cf3a776ba00e7c
2476312,1496895420,0.160597,1,88cf3a776ba00e7c
2476313,1496895480,0.160393,1,88cf3a776ba00e7c


In [5]:
df = df.sort_values(['KPI ID', 'timestamp']).reset_index(drop=True)

In [6]:
df

,timestamp,value,label,KPI ID
0,1493568000,1.901639,0,02e99bd4f6cfb33f
1,1493568060,1.786885,0,02e99bd4f6cfb33f
2,1493568120,2.000000,0,02e99bd4f6cfb33f
3,1493568180,1.885246,0,02e99bd4f6cfb33f
4,1493568240,1.819672,0,02e99bd4f6cfb33f
...,...,...,...,...
2476310,1500146040,3070.000000,0,e0770391decc44ce
2476311,1500146100,3019.000000,0,e0770391decc44ce
2476312,1500146160,3508.000000,0,e0770391decc44ce
2476313,1500146220,3341.000000,0,e0770391decc44ce


In [7]:
kpi_freq = (
    df.groupby('KPI ID')['timestamp']
    .diff()
    .dropna()
    .groupby(df['KPI ID'])
    .agg(lambda x: x.mode()[0])
    .astype(int)
)

results = []
for kpi_id in df['KPI ID'].unique():
    group = df[df['KPI ID'] == kpi_id].copy()
    freq  = kpi_freq[kpi_id]
    group['datetime'] = pd.to_datetime(group['timestamp'], unit='s')
    
    group = (group
        .set_index('datetime')
        .resample(f'{freq}s')
        .first()
        .reset_index()
    )
    group['KPI ID'] = kpi_id
    results.append(group)

df = pd.concat(results, ignore_index=True)
df = df.sort_values(['KPI ID', 'datetime']).reset_index(drop=True)

In [8]:
df

,datetime,timestamp,value,label,KPI ID
0,2017-04-30 16:00:00,1.493568e+09,1.901639,0.0,02e99bd4f6cfb33f
1,2017-04-30 16:01:00,1.493568e+09,1.786885,0.0,02e99bd4f6cfb33f
2,2017-04-30 16:02:00,1.493568e+09,2.000000,0.0,02e99bd4f6cfb33f
3,2017-04-30 16:03:00,1.493568e+09,1.885246,0.0,02e99bd4f6cfb33f
4,2017-04-30 16:04:00,1.493568e+09,1.819672,0.0,02e99bd4f6cfb33f
...,...,...,...,...,...
2532567,2017-07-15 19:14:00,1.500146e+09,3070.000000,0.0,e0770391decc44ce
2532568,2017-07-15 19:15:00,1.500146e+09,3019.000000,0.0,e0770391decc44ce
2532569,2017-07-15 19:16:00,1.500146e+09,3508.000000,0.0,e0770391decc44ce
2532570,2017-07-15 19:17:00,1.500146e+09,3341.000000,0.0,e0770391decc44ce


In [9]:
df = df.drop(columns=['timestamp'])
df = df.rename(columns={'datetime': 'timestamp'})

print(df.head())

            timestamp     value  label            KPI ID
0 2017-04-30 16:00:00  1.901639    0.0  02e99bd4f6cfb33f
1 2017-04-30 16:01:00  1.786885    0.0  02e99bd4f6cfb33f
2 2017-04-30 16:02:00  2.000000    0.0  02e99bd4f6cfb33f
3 2017-04-30 16:03:00  1.885246    0.0  02e99bd4f6cfb33f
4 2017-04-30 16:04:00  1.819672    0.0  02e99bd4f6cfb33f


In [10]:
print(df.isna().sum())

timestamp        0
value        56257
label        56257
KPI ID           0
dtype: int64


In [11]:
df['value'] = df.groupby('KPI ID')['value'].ffill(limit=5)
print(df[df['value'].isna()].groupby('KPI ID').size().sort_values(ascending=False))

KPI ID
02e99bd4f6cfb33f    3086
9bd90500bfd11edb    3030
c58bfcbacb2822d1    2988
a5bf5d65261d859a    2973
1c35dbf57f55f5e4    2817
09513ae3e75778a3    2708
da403e4e3f87c9e0    2639
18fbb1d5a5dc099d    2538
8c892e5525f3e491    1648
e0770391decc44ce    1640
07927a9a18fa19ae     987
cff6d3c01e6a6bfa     390
71595dd7171f4540     330
a40b1df87e3f1c87     309
7c189dd36f048a6c     308
8bef9af9a922e0b3     308
affb01ca2b4f0b45     308
54e8a140f6237526      23
b3b2e6d1a791d63a      23
40e25005ff8992bd       1
dtype: int64


In [12]:
lag_minutes = [1, 5, 10, 60]

for minutes in lag_minutes:
    col = f'lag_{minutes}m'
    df[col] = float('nan')
    for kpi_id in df['KPI ID'].unique():
        freq  = int(kpi_freq[kpi_id])
        steps = int(minutes * 60 / freq)
        mask  = df['KPI ID'] == kpi_id
        df.loc[mask, col] = df.loc[mask, 'value'].shift(steps)

lag_cols = ['timestamp', 'KPI ID', 'value', 'lag_1m', 'lag_5m', 'lag_10m', 'lag_60m']
print(df[lag_cols].head(10))

            timestamp            KPI ID     value    lag_1m    lag_5m  \
0 2017-04-30 16:00:00  02e99bd4f6cfb33f  1.901639       NaN       NaN   
1 2017-04-30 16:01:00  02e99bd4f6cfb33f  1.786885  1.901639       NaN   
2 2017-04-30 16:02:00  02e99bd4f6cfb33f  2.000000  1.786885       NaN   
3 2017-04-30 16:03:00  02e99bd4f6cfb33f  1.885246  2.000000       NaN   
4 2017-04-30 16:04:00  02e99bd4f6cfb33f  1.819672  1.885246       NaN   
5 2017-04-30 16:05:00  02e99bd4f6cfb33f  1.885246  1.819672  1.901639   
6 2017-04-30 16:06:00  02e99bd4f6cfb33f  1.885246  1.885246  1.786885   
7 2017-04-30 16:07:00  02e99bd4f6cfb33f  1.934426  1.885246  2.000000   
8 2017-04-30 16:08:00  02e99bd4f6cfb33f  1.967213  1.934426  1.885246   
9 2017-04-30 16:09:00  02e99bd4f6cfb33f  1.950820  1.967213  1.819672   

   lag_10m  lag_60m  
0      NaN      NaN  
1      NaN      NaN  
2      NaN      NaN  
3      NaN      NaN  
4      NaN      NaN  
5      NaN      NaN  
6      NaN      NaN  
7      NaN      NaN 

In [13]:
roll_minutes = [5, 15, 60]

for minutes in roll_minutes:
    for stat in ['mean', 'std', 'min', 'max']:
        col = f'roll_{stat}_{minutes}m'
        df[col] = float('nan')
        for kpi_id in df['KPI ID'].unique():
            freq  = int(kpi_freq[kpi_id])
            steps = int(minutes * 60 / freq)
            mask  = df['KPI ID'] == kpi_id
            df.loc[mask, col] = (
                df.loc[mask, 'value']
                .rolling(steps)
                .agg(stat)
            )
roll_cols = ['timestamp', 'KPI ID', 'value', 
             'roll_mean_5m', 'roll_std_5m', 'roll_min_5m', 'roll_max_5m']
print(df[roll_cols].head(10))

            timestamp            KPI ID     value  roll_mean_5m  roll_std_5m  \
0 2017-04-30 16:00:00  02e99bd4f6cfb33f  1.901639           NaN          NaN   
1 2017-04-30 16:01:00  02e99bd4f6cfb33f  1.786885           NaN          NaN   
2 2017-04-30 16:02:00  02e99bd4f6cfb33f  2.000000           NaN          NaN   
3 2017-04-30 16:03:00  02e99bd4f6cfb33f  1.885246           NaN          NaN   
4 2017-04-30 16:04:00  02e99bd4f6cfb33f  1.819672      1.878689     0.082458   
5 2017-04-30 16:05:00  02e99bd4f6cfb33f  1.885246      1.875410     0.081639   
6 2017-04-30 16:06:00  02e99bd4f6cfb33f  1.885246      1.895082     0.065163   
7 2017-04-30 16:07:00  02e99bd4f6cfb33f  1.934426      1.881967     0.040819   
8 2017-04-30 16:08:00  02e99bd4f6cfb33f  1.967213      1.898361     0.056074   
9 2017-04-30 16:09:00  02e99bd4f6cfb33f  1.950820      1.924590     0.037741   

   roll_min_5m  roll_max_5m  
0          NaN          NaN  
1          NaN          NaN  
2          NaN          NaN  

In [15]:
df

,timestamp,value,label,KPI ID,lag_1m,lag_5m,lag_10m,lag_60m,roll_mean_5m,roll_std_5m,roll_min_5m,roll_max_5m,roll_mean_15m,roll_std_15m,roll_min_15m,roll_max_15m,roll_mean_60m,roll_std_60m,roll_min_60m,roll_max_60m
0,2017-04-30 16:00:00,1.901639,0.0,02e99bd4f6cfb33f,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-04-30 16:01:00,1.786885,0.0,02e99bd4f6cfb33f,1.901639,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2017-04-30 16:02:00,2.000000,0.0,02e99bd4f6cfb33f,1.786885,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2017-04-30 16:03:00,1.885246,0.0,02e99bd4f6cfb33f,2.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2017-04-30 16:04:00,1.819672,0.0,02e99bd4f6cfb33f,1.885246,NaN,NaN,NaN,1.878689,0.082458,1.786885,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2532567,2017-07-15 19:14:00,3070.000000,0.0,e0770391decc44ce,3084.000000,3017.0,3296.0,3778.0,3241.200000,222.170430,3070.000000,3485.0,3270.533333,189.499290,3017.0,3653.0,3403.483333,185.205016,3017.0,3850.0
2532568,2017-07-15 19:15:00,3019.000000,0.0,e0770391decc44ce,3070.000000,3485.0,3392.0,3507.0,3148.000000,189.698445,3019.000000,3484.0,3259.666667,199.355987,3017.0,3653.0,3395.350000,191.200297,3017.0,3850.0
2532569,2017-07-15 19:16:00,3508.000000,0.0,e0770391decc44ce,3019.000000,3484.0,3266.0,3763.0,3152.800000,200.331475,3019.000000,3508.0,3266.800000,206.558605,3017.0,3653.0,3391.100000,185.642969,3017.0,3850.0
2532570,2017-07-15 19:17:00,3341.000000,0.0,e0770391decc44ce,3508.000000,3083.0,3073.0,3677.0,3204.400000,210.811053,3019.000000,3508.0,3265.600000,206.036335,3017.0,3653.0,3385.500000,181.902609,3017.0,3850.0


In [ ]:
roc_minutes = [1, 5, 10, 60]

for minutes in roc_minutes:
    diff_col = f'diff_{minutes}m'
    pct_col  = f'pct_{minutes}m'
    df[diff_col] = float('nan')
    df[pct_col]  = float('nan')
    
    for kpi_id in df['KPI ID'].unique():
        freq  = int(kpi_freq[kpi_id])
        steps = int(minutes * 60 / freq)
        mask  = df['KPI ID'] == kpi_id
        
        df.loc[mask, diff_col] = df.loc[mask, 'value'].diff(steps)
        
        df.loc[mask, pct_col]  = df.loc[mask, 'value'].pct_change(steps)

pct_cols = [f'pct_{m}m' for m in roc_minutes]
df[pct_cols] = df[pct_cols].replace([float('inf'), float('-inf')], float('nan'))
df[pct_cols] = df[pct_cols].clip(-10, 10)

roc_cols = ['timestamp', 'KPI ID', 'value', 'diff_1m', 'pct_1m', 'diff_5m', 'pct_5m']
print(df[roc_cols].head(10))

            timestamp            KPI ID     value   diff_1m    pct_1m  \
0 2017-04-30 16:00:00  02e99bd4f6cfb33f  1.901639       NaN       NaN   
1 2017-04-30 16:01:00  02e99bd4f6cfb33f  1.786885 -0.114754 -0.060345   
2 2017-04-30 16:02:00  02e99bd4f6cfb33f  2.000000  0.213115  0.119266   
3 2017-04-30 16:03:00  02e99bd4f6cfb33f  1.885246 -0.114754 -0.057377   
4 2017-04-30 16:04:00  02e99bd4f6cfb33f  1.819672 -0.065574 -0.034783   
5 2017-04-30 16:05:00  02e99bd4f6cfb33f  1.885246  0.065574  0.036036   
6 2017-04-30 16:06:00  02e99bd4f6cfb33f  1.885246  0.000000  0.000000   
7 2017-04-30 16:07:00  02e99bd4f6cfb33f  1.934426  0.049180  0.026087   
8 2017-04-30 16:08:00  02e99bd4f6cfb33f  1.967213  0.032787  0.016949   
9 2017-04-30 16:09:00  02e99bd4f6cfb33f  1.950820 -0.016393 -0.008333   

    diff_5m    pct_5m  
0       NaN       NaN  
1       NaN       NaN  
2       NaN       NaN  
3       NaN       NaN  
4       NaN       NaN  
5 -0.016393 -0.008621  
6  0.098361  0.055046  
7 -0

In [20]:
df['ratio_value_to_mean_5m']  = df['value'] / df['roll_mean_5m'].replace(0, float('nan'))
df['ratio_value_to_mean_15m'] = df['value'] / df['roll_mean_15m'].replace(0, float('nan'))
df['ratio_value_to_mean_60m'] = df['value'] / df['roll_mean_60m'].replace(0, float('nan'))

df['zscore_5m']  = (df['value'] - df['roll_mean_5m'])  / df['roll_std_5m'].replace(0, float('nan'))
df['zscore_15m'] = (df['value'] - df['roll_mean_15m']) / df['roll_std_15m'].replace(0, float('nan'))
df['zscore_60m'] = (df['value'] - df['roll_mean_60m']) / df['roll_std_60m'].replace(0, float('nan'))

df['ratio_minmax_5m']  = (df['value'] - df['roll_min_5m'])  / (df['roll_max_5m']  - df['roll_min_5m']).replace(0, float('nan'))
df['ratio_minmax_60m'] = (df['value'] - df['roll_min_60m']) / (df['roll_max_60m'] - df['roll_min_60m']).replace(0, float('nan'))

# Kiểm tra
ratio_cols = ['timestamp', 'KPI ID', 'value',
              'ratio_value_to_mean_5m', 'zscore_5m', 'ratio_minmax_5m']
print(df[ratio_cols].head(10))

            timestamp            KPI ID     value  ratio_value_to_mean_5m  \
0 2017-04-30 16:00:00  02e99bd4f6cfb33f  1.901639                     NaN   
1 2017-04-30 16:01:00  02e99bd4f6cfb33f  1.786885                     NaN   
2 2017-04-30 16:02:00  02e99bd4f6cfb33f  2.000000                     NaN   
3 2017-04-30 16:03:00  02e99bd4f6cfb33f  1.885246                     NaN   
4 2017-04-30 16:04:00  02e99bd4f6cfb33f  1.819672                0.968586   
5 2017-04-30 16:05:00  02e99bd4f6cfb33f  1.885246                1.005245   
6 2017-04-30 16:06:00  02e99bd4f6cfb33f  1.885246                0.994810   
7 2017-04-30 16:07:00  02e99bd4f6cfb33f  1.934426                1.027875   
8 2017-04-30 16:08:00  02e99bd4f6cfb33f  1.967213                1.036269   
9 2017-04-30 16:09:00  02e99bd4f6cfb33f  1.950820                1.013629   

   zscore_5m  ratio_minmax_5m  
0        NaN              NaN  
1        NaN              NaN  
2        NaN              NaN  
3        NaN            

In [22]:
import numpy as np

df['hour_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.hour / 24)

df['dow_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.dayofweek / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.dayofweek / 7)

df['is_weekend'] = (df['timestamp'].dt.dayofweek >= 5).astype(int)

time_cols = ['timestamp', 'KPI ID', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend']
print(df[time_cols].head(10))

            timestamp            KPI ID  hour_sin  hour_cos   dow_sin  \
0 2017-04-30 16:00:00  02e99bd4f6cfb33f -0.866025      -0.5 -0.781831   
1 2017-04-30 16:01:00  02e99bd4f6cfb33f -0.866025      -0.5 -0.781831   
2 2017-04-30 16:02:00  02e99bd4f6cfb33f -0.866025      -0.5 -0.781831   
3 2017-04-30 16:03:00  02e99bd4f6cfb33f -0.866025      -0.5 -0.781831   
4 2017-04-30 16:04:00  02e99bd4f6cfb33f -0.866025      -0.5 -0.781831   
5 2017-04-30 16:05:00  02e99bd4f6cfb33f -0.866025      -0.5 -0.781831   
6 2017-04-30 16:06:00  02e99bd4f6cfb33f -0.866025      -0.5 -0.781831   
7 2017-04-30 16:07:00  02e99bd4f6cfb33f -0.866025      -0.5 -0.781831   
8 2017-04-30 16:08:00  02e99bd4f6cfb33f -0.866025      -0.5 -0.781831   
9 2017-04-30 16:09:00  02e99bd4f6cfb33f -0.866025      -0.5 -0.781831   

   dow_cos  is_weekend  
0  0.62349           1  
1  0.62349           1  
2  0.62349           1  
3  0.62349           1  
4  0.62349           1  
5  0.62349           1  
6  0.62349           

In [23]:
print(df.columns.tolist())
print(df.shape)
print(df.isna().sum())

['timestamp', 'value', 'label', 'KPI ID', 'lag_1m', 'lag_5m', 'lag_10m', 'lag_60m', 'roll_mean_5m', 'roll_std_5m', 'roll_min_5m', 'roll_max_5m', 'roll_mean_15m', 'roll_std_15m', 'roll_min_15m', 'roll_max_15m', 'roll_mean_60m', 'roll_std_60m', 'roll_min_60m', 'roll_max_60m', 'diff_1m', 'pct_1m', 'diff_5m', 'pct_5m', 'diff_10m', 'pct_10m', 'diff_60m', 'pct_60m', 'stl_residual', 'ratio_value_to_mean_5m', 'ratio_value_to_mean_15m', 'ratio_value_to_mean_60m', 'zscore_5m', 'zscore_15m', 'zscore_60m', 'ratio_minmax_5m', 'ratio_minmax_60m', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend']
(2532572, 42)
timestamp                        0
value                        29054
label                        56257
KPI ID                           0
lag_1m                       29073
lag_5m                       29156
lag_10m                      29258
lag_60m                      30278
roll_mean_5m                 29802
roll_std_5m                  92803
roll_min_5m                  29802
ro

In [24]:
df = df.drop(columns=['stl_residual'])

In [25]:
print(df['pct_1m'].isna().sum())
print((df['pct_1m'] == float('inf')).sum())

160966
0


In [26]:
pct_cols = ['pct_1m', 'pct_5m', 'pct_10m', 'pct_60m']
df[pct_cols] = df[pct_cols].fillna(0)

In [27]:
df = df.dropna()

print(df.shape)
print(df.isna().sum().sum())

(2265662, 41)
0


In [28]:
df

,timestamp,value,label,KPI ID,lag_1m,lag_5m,lag_10m,lag_60m,roll_mean_5m,roll_std_5m,...,zscore_5m,zscore_15m,zscore_60m,ratio_minmax_5m,ratio_minmax_60m,hour_sin,hour_cos,dow_sin,dow_cos,is_weekend
60,2017-04-30 17:00:00,2.475410,0.0,02e99bd4f6cfb33f,2.557377,2.245902,2.426229,1.901639,2.403279,0.177171,...,0.407128,1.178217,2.155718,0.772727,0.893617,-0.965926,-0.258819,-0.781831,0.623490,1
61,2017-04-30 17:01:00,2.442623,0.0,02e99bd4f6cfb33f,2.475410,2.229508,2.016393,1.786885,2.445902,0.148177,...,-0.022127,0.894737,1.899095,0.681818,0.847826,-0.965926,-0.258819,-0.781831,0.623490,1
62,2017-04-30 17:02:00,2.377049,0.0,02e99bd4f6cfb33f,2.442623,2.196721,1.967213,2.000000,2.481967,0.077415,...,-1.355275,0.490399,1.485107,0.000000,0.760870,-0.965926,-0.258819,-0.781831,0.623490,1
63,2017-04-30 17:03:00,2.344262,0.0,02e99bd4f6cfb33f,2.377049,2.557377,2.049180,1.885246,2.439344,0.083911,...,-1.133123,0.259350,1.266675,0.000000,0.717391,-0.965926,-0.258819,-0.781831,0.623490,1
64,2017-04-30 17:04:00,2.098361,0.0,02e99bd4f6cfb33f,2.344262,2.557377,2.295082,1.819672,2.347541,0.148630,...,-1.676516,-0.972044,-0.060484,0.000000,0.391304,-0.965926,-0.258819,-0.781831,0.623490,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2532567,2017-07-15 19:14:00,3070.000000,0.0,e0770391decc44ce,3084.000000,3017.000000,3296.000000,3778.000000,3241.200000,222.170430,...,-0.770580,-1.058227,-1.800617,0.000000,0.063625,-0.965926,0.258819,-0.974928,-0.222521,1
2532568,2017-07-15 19:15:00,3019.000000,0.0,e0770391decc44ce,3070.000000,3485.000000,3392.000000,3507.000000,3148.000000,189.698445,...,-0.680027,-1.207221,-1.968355,0.000000,0.002401,-0.965926,0.258819,-0.974928,-0.222521,1
2532569,2017-07-15 19:16:00,3508.000000,0.0,e0770391decc44ce,3019.000000,3484.000000,3266.000000,3763.000000,3152.800000,200.331475,...,1.773061,1.167707,0.629703,1.000000,0.589436,-0.965926,0.258819,-0.974928,-0.222521,1
2532570,2017-07-15 19:17:00,3341.000000,0.0,e0770391decc44ce,3508.000000,3083.000000,3073.000000,3677.000000,3204.400000,210.811053,...,0.647974,0.365955,-0.244636,0.658487,0.388956,-0.965926,0.258819,-0.974928,-0.222521,1


In [30]:
df['is_weekend'].value_counts()

is_weekend
0    1624187
1     641475
Name: count, dtype: int64

In [36]:
import os
os.makedirs('../../Week2/Data', exist_ok=True)
df.to_csv('../../Week2/Data/train_fe.csv', index=False)
print(df.shape)

(2265662, 41)
